# ⚽ Visión por computadora con YOLO: analizando la final del Mundial 2022 — versión del alumno ✏️

**Taller práctico** · Argentina 🇦🇷 vs Francia 🇫🇷 · Final de la Copa Mundial de la FIFA, Qatar 2022 · ~15 s de una toma abierta continua

En este notebook vamos a construir paso a paso un sistema que, a partir de un video de la transmisión de TV:

1. **Detecta** jugadores, árbitros y balón con YOLO
2. **Sigue** a cada jugador a lo largo del video (tracking)
3. **Asigna equipo** a cada jugador según el color de su camiseta (K-Means)
4. **Calcula la posesión** del balón por equipo
5. *(Bonus)* Estima el **movimiento de cámara**, convierte píxeles a **metros** y mide **velocidad y distancia** de cada jugador

Basado en [abdullahtarek/football_analysis](https://github.com/abdullahtarek/football_analysis), adaptado a video de transmisión.

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

## ✏️ Cómo usar esta versión

Este notebook tiene **7 huecos** marcados con ✏️ donde tú escribes el código (son de 1 a 3 líneas).

1. Lee la instrucción y reemplaza cada `_____` por tu respuesta.
2. Ejecuta la celda y luego la de **verificación** que está debajo: te dirá ✅ o ❌.
3. ¿Atorado? Abre la **💡 pista**. Si sigue sin salir, abre **👀 Ver solución** y cópiala: lo importante es entenderla.

> Si ves `NameError: name '_____' is not defined`, te falta llenar un hueco de esa celda.

> 💾 **Antes de empezar:** `Archivo → Guardar una copia en Drive`. Así te quedas con tu versión al terminar.

## 0. Preparación

In [ ]:
# ⚙️ Configuración del taller (el instructor llena estos valores)
REPO_URL = "https://github.com/JosephLeyva/proxy_football_analysis"   # repo con el código adaptado
CLIP_DRIVE_ID = "https://drive.google.com/file/d/1BGFoI8J1OtJRRL4_TZ1pHEIrOjaBjQyq/view?usp=sharing"    # enlace/ID de Drive de fifa_wide.mp4 (ARG vs FRA, 1080p, 387 frames).
MODEL_DRIVE_ID = "1DC2kCygbBWUKheQ_9cFziCsYVSRw6axK"   # best.pt del repo original

# True = usar los resultados precalculados (stubs) en vez de correr YOLO sobre todos los frames del clip
USE_STUBS = True


In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !nvidia-smi --query-gpu=name,memory.total --format=csv
    if not os.path.exists("/content/proyecto"):
        !git clone -q {REPO_URL} /content/proyecto
    %cd /content/proyecto
    !pip install -q -r requirements.txt
else:
    # Ejecucion local: el notebook vive en taller/
    if os.path.basename(os.getcwd()) == "taller":
        os.chdir("..")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
import sys, config

if not os.path.exists(config.MODEL_PATH):
    !{sys.executable} -m gdown -q --fuzzy {MODEL_DRIVE_ID} -O {config.MODEL_PATH}

if not os.path.exists(config.VIDEO_PATH):
    if CLIP_DRIVE_ID:
        # --fuzzy: acepta la URL completa de Drive y resuelve el aviso de "archivo grande" de Google
        !{sys.executable} -m gdown -q --fuzzy {CLIP_DRIVE_ID} -O {config.VIDEO_PATH}
    elif IN_COLAB:
        from google.colab import files
        print("Sube el archivo del clip")
        uploaded = files.upload()
        os.rename(next(iter(uploaded)), config.VIDEO_PATH)

# Si el enlace de Drive no es publico o el archivo es grande, gdown a veces guarda una pagina de
# aviso en vez del archivo real: existe pero pesa unos KB en lugar de varios MB. Lo detectamos aqui
# en vez de fallar mas adelante con un error confuso de OpenCV.
for path in (config.MODEL_PATH, config.VIDEO_PATH):
    if os.path.exists(path) and os.path.getsize(path) < 1_000_000:
        raise RuntimeError(
            f"{path} se descargo mal ({os.path.getsize(path)} bytes). "
            "Revisa que el archivo de Drive sea publico ('Cualquiera con el enlace') y vuelve a "
            "correr esta celda (borra antes el archivo incompleto con !rm -f " + path + ")."
        )

print("Modelo:", os.path.exists(config.MODEL_PATH), "| Video:", os.path.exists(config.VIDEO_PATH))


In [ ]:
# Utilidades para mostrar imagenes y videos dentro del notebook
import cv2
import numpy as np
import matplotlib.pyplot as plt
from base64 import b64encode
from IPython.display import HTML

def show(*images, titles=None, width=16):
    fig, axes = plt.subplots(1, len(images), figsize=(width, width*9/16/len(images)+1))
    axes = np.atleast_1d(axes)
    for i, (ax, img) in enumerate(zip(axes, images)):
        is_color = img.ndim == 3
        # vmin/vmax explicitos: sin esto, una imagen de un solo canal con valores constantes
        # (p.ej. una mascara binaria toda en 0 o toda en 255) se ve negra sin importar su valor real,
        # porque matplotlib no tiene rango para normalizar y cae al extremo inferior del colormap.
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if is_color else img,
                  cmap=None if is_color else "gray",
                  vmin=None if is_color else 0, vmax=None if is_color else 255)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

def show_video(path, width=900):
    data = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

# Verificación de los huecos ✏️ (no la modifiques)
def revisar(n, condicion, si_falla=""):
    try:
        ok = bool(condicion())
    except NameError as e:
        print(f"⚠️ Hueco {n}: falta ejecutar la celda del hueco (o algo quedó sin llenar): {e}")
        return
    except Exception as e:
        ok = False
        si_falla = f"{si_falla} (error: {type(e).__name__}: {e})"
    print(f"✅ Hueco {n}: ¡correcto!" if ok else f"❌ Hueco {n}: todavía no. {si_falla}")


In [ ]:
from utils import read_video, get_video_fps

frames = read_video(config.VIDEO_PATH)
fps = get_video_fps(config.VIDEO_PATH)
print(f"{len(frames)} frames · {fps:.0f} fps · {len(frames)/fps:.0f} s · resolución {frames[0].shape[1]}x{frames[0].shape[0]}")
show(frames[0], titles=["Frame 0"])

# Los stubs (resultados precalculados) solo sirven para ESTE video: si el clip descargado es otro,
# todo correría sin error pero con resultados sin sentido. Lo verificamos con el número de frames.
import pickle
if USE_STUBS and os.path.exists(config.TRACK_STUB_PATH):
    with open(config.TRACK_STUB_PATH, "rb") as f:
        stub_frames = len(pickle.load(f)["players"])
    if stub_frames != len(frames):
        raise RuntimeError(
            f"El video tiene {len(frames)} frames pero {config.TRACK_STUB_PATH} es de un video de {stub_frames}. "
            "Revisa que CLIP_DRIVE_ID apunte al mismo clip que config.VIDEO_PATH.")


> 💡 **Ojo con la memoria:** `read_video` guarda *todos* los frames en RAM. Un frame de 1080p pesa ~6 MB,
> así que los 387 frames de este clip son ~2.4 GB. Un minuto a 1080p serían más de 10 GB y Colab (~12 GB) no aguanta;
> por eso trabajamos con un clip corto de una sola toma abierta, sin cortes a repeticiones ni primeros planos.
> YOLO reescala cada imagen a 640 px, así que la resolución original no cambia mucho la precisión.

## 1. YOLO "de fábrica" (COCO) · ~15 min

YOLO (*You Only Look Once*) mira la imagen **una sola vez** y predice todas las cajas y clases al mismo tiempo.
Los modelos preentrenados de Ultralytics conocen las 80 clases del dataset COCO: `person`, `sports ball`, `car`...

¿Qué tan bien funciona en fútbol sin entrenarlo?

### ✏️ Hueco 1: Carga YOLO y haz tu primera detección

Carga el modelo **nano** de YOLO11 (entrenado con COCO) y pásale el **primer frame** del video.

<details><summary>💡 Pista</summary>

El modelo se llama `"yolo11n.pt"` (con comillas: es un nombre de archivo). Los frames están en la lista `frames`; el primero es el índice 0.

</details>

In [ ]:
from ultralytics import YOLO

coco_model = YOLO(_____)          # ✏️ HUECO 1a: el modelo nano de YOLO11
result = coco_model.predict(_____, conf=0.25, verbose=False)[0]   # ✏️ HUECO 1b: el primer frame del video

from collections import Counter
print(Counter(result.names[int(c)] for c in result.boxes.cls))
show(result.plot(line_width=2, font_size=10), titles=["YOLO11n entrenado con COCO"])

In [ ]:
# Verificación del hueco 1
revisar(1, lambda: len(result.boxes) > 0 and "person" in result.names.values(), '¿Usaste el nombre del modelo entre comillas y frames[0]?')

<details><summary>👀 Ver solución del hueco 1</summary>

```python
coco_model = YOLO("yolo11n.pt")          # n = nano, el mas pequeño y rapido
result = coco_model.predict(frames[0], conf=0.25, verbose=False)[0]
```

</details>

**Observa:**
- Todos son `person`: YOLO no sabe distinguir **jugador, portero o árbitro**.
- El balón (`sports ball`) es diminuto: a veces no lo detecta.
- En otros frames puede detectar gente del público o del cuerpo técnico.

➡️ Necesitamos un modelo **especializado** en fútbol.

🧪 **Reto:** prueba `yolo11x.pt` (el más grande) o baja `conf` a `0.1`. ¿Aparece el balón? ¿Cuánto más tarda?

## 2. Un YOLO especializado en fútbol · ~15 min

`models/best.pt` es un YOLOv5x reentrenado (*fine-tuning*) con el dataset
[football-players-detection de Roboflow](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc):
imágenes de partidos etiquetadas con 4 clases.

Así se entrenó (tarda bastante incluso con GPU, **no lo corras ahora**):

```python
!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="TU_API_KEY")
dataset = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc").version(1).download("yolov5")

!yolo task=detect mode=train model=yolov5x.pt data={dataset.location}/data.yaml epochs=100 imgsz=640
```

### ✏️ Hueco 2: Carga el modelo especializado en fútbol

Ahora carga nuestro YOLO reentrenado con imágenes de fútbol. Su ruta ya está guardada en la configuración del proyecto.

<details><summary>💡 Pista</summary>

Busca en `config` una variable con la ruta del modelo: `config.MODEL_PATH`.

</details>

In [ ]:
import time

model = YOLO(_____)   # ✏️ HUECO 2: la ruta del modelo de fútbol
print("Clases:", model.names)

t = time.time()
result = model.predict(frames[0], conf=0.1, verbose=False)[0]
print(f"Tiempo: {time.time()-t:.2f} s")
print(Counter(result.names[int(c)] for c in result.boxes.cls))
show(result.plot(line_width=2, font_size=10), titles=["best.pt: ball / goalkeeper / player / referee"])

In [ ]:
# Verificación del hueco 2
revisar(2, lambda: set(model.names.values()) == {"ball", "goalkeeper", "player", "referee"}, 'El modelo correcto tiene 4 clases: ball, goalkeeper, player y referee.')

<details><summary>👀 Ver solución del hueco 2</summary>

```python
model = YOLO(config.MODEL_PATH)
```

</details>

Ahora sí distingue **jugadores, árbitros y balón** 🎉 (¡incluso el juez de línea!). Pero:
- Cada frame es independiente: el jugador de la izquierda en el frame 0 no "sabe" que es el mismo del frame 1.
- A veces confunde clases o detecta cosas fuera de la cancha.

🧪 **Reto:** corre el modelo sobre `frames[100]` y `frames[300]`. ¿Hay algún error de clase?


## 3. Tracking: seguir a cada jugador · ~20 min

El *tracker* (**ByteTrack**) une las detecciones de frames consecutivos: predice hacia dónde se moverá cada caja
y la empareja con la detección más cercana. Así cada jugador conserva un **ID** durante el video.

Además, en `trackers/tracker.py` agregamos dos mejoras para video de TV:
1. **Máscara de cancha**: se descartan detecciones cuyos pies no están sobre el pasto (tribunas, anuncios, marcador).
2. **Voto por mayoría**: si un track fue "jugador" en 95 frames y "árbitro" en 5, se queda como jugador.

> 💡 **Sobre la máscara de cancha:** es un umbral de color (verde = pasto). Si las vallas publicitarias LED
> del estadio son del mismo verde que el pasto y lo tocan directamente en la imagen (como en este clip), el
> umbral no puede separarlas — comparten el mismo color, no hay forma de distinguirlas solo por HSV. Por eso
> la máscara de abajo cubre más que la cancha real. No es un bug de ajuste de parámetros: es una limitación
> física del método. La exclusión definitiva de gente fuera de la cancha llega más adelante (sección 6), una
> vez que convertimos posiciones a **metros** con la homografía de la cancha.


In [ ]:
from utils import get_pitch_mask

mask = get_pitch_mask(frames[0])
print(f"Cobertura de la mascara: {mask.mean()*100:.0f}% del frame")

# Solo para esta demo: las vallas LED verdes tocan el pasto y comparten su color exacto, asi que
# el umbral de color las incluye tambien. Ignorar el ~22% superior (donde esta la tribuna/vallas)
# da una silueta de cancha creible PARA ESTA TOMA especifica -- no se hace asi dentro de
# get_pitch_mask porque en frames con zoom la camara a veces encuadra solo pasto hasta el borde
# superior, y recortar ahi cortaria jugadores reales.
h = frames[0].shape[0]
top_crop = int(h * 0.22)
mask_demo = mask.copy()
mask_demo[:top_crop] = 0

show(frames[0], mask*255, mask_demo*255,
     titles=["Frame",
             f"Mascara real ({mask.mean()*100:.0f}% del frame: incluye las vallas LED)",
             "Demo: ignorando el 22% superior"])


In [ ]:
from trackers import Tracker

tracker = Tracker(config.MODEL_PATH, conf=config.DETECTION_CONF,
                  overlay_boxes=config.OVERLAY_BOXES, ball_control_box=config.BALL_CONTROL_BOX)

# Con USE_STUBS=True se cargan los resultados guardados; si no, corre YOLO + ByteTrack (varios minutos en T4)
tracks = tracker.get_object_tracks(frames, read_from_stub=USE_STUBS, stub_path=config.TRACK_STUB_PATH)
tracker.add_position_to_tracks(tracks)

print("Estructura:", {k: f"{len(v)} frames" for k, v in tracks.items()})
print("Frame 0, jugador de ejemplo:", next(iter(tracks["players"][0].items())))

In [ ]:
# Dibujamos IDs sobre algunos frames (copias, para no modificar los originales)
def draw_tracks(frame_num):
    frame = frames[frame_num].copy()
    for track_id, player in tracks["players"][frame_num].items():
        tracker.draw_ellipse(frame, player["bbox"], (0, 0, 255), track_id)
    for _, referee in tracks["referees"][frame_num].items():
        tracker.draw_ellipse(frame, referee["bbox"], (255, 255, 255))
    return frame

show(draw_tracks(150), draw_tracks(300), titles=["Frame 150", "Frame 300 (~6 s después): mismos IDs"])

### ✏️ Hueco 3: Cuenta jugadores e IDs

`tracks["players"]` es una **lista** con un elemento por frame. Cada elemento es un **diccionario** `{track_id: datos_del_jugador}`. Calcula:
- `unique_ids`: el conjunto de todos los IDs que aparecen en el video.
- `per_frame`: una lista con cuántos jugadores hay en cada frame.

<details><summary>💡 Pista</summary>

Recorrer un diccionario con `for tid in frame` te da sus llaves (los IDs). `len(f)` dice cuántas llaves tiene `f`.

</details>

In [ ]:
unique_ids = {_____ for frame in tracks["players"] for tid in frame}   # ✏️ HUECO 3a: el ID de cada track
per_frame = [_____ for f in tracks["players"]]                          # ✏️ HUECO 3b: cuántos jugadores hay en el frame f

plt.figure(figsize=(12, 3))
plt.plot(per_frame)
plt.xlabel("frame"); plt.ylabel("jugadores detectados")
plt.title(f"Jugadores por frame · IDs únicos en todo el clip: {len(unique_ids)}")
plt.show()

In [ ]:
# Verificación del hueco 3
revisar(3, lambda: len(per_frame) == len(frames) and per_frame[0] == len(tracks["players"][0]) and unique_ids == {t for fr in tracks["players"] for t in fr}, 'unique_ids debe tener los IDs (tid) y per_frame un número por frame.')

<details><summary>👀 Ver solución del hueco 3</summary>

```python
unique_ids = {tid for frame in tracks["players"] for tid in frame}
per_frame = [len(f) for f in tracks["players"]]
```

</details>

En la cancha hay 22 jugadores, pero la cámara solo muestra una parte. Si hay **más IDs únicos** que jugadores
que entraron a cuadro, es porque el tracker perdió a alguien (oclusión, salió de cuadro) y le dio un ID nuevo:
es un *ID switch*, el problema clásico del tracking.

🧪 **Reto:** busca un ID que aparezca solo unos pocos frames. ¿Qué pasó ahí?

## 4. ¿De qué equipo es cada jugador? K-Means · ~20 min

Nadie etiquetó los equipos: los vamos a descubrir con **aprendizaje no supervisado**.

**Paso A:** para cada jugador, recortamos la mitad superior de su caja (la camiseta) y agrupamos sus píxeles
en 2 colores con K-Means: uno es el fondo (pasto) y otro la camiseta. El fondo es el color de las esquinas.

### ✏️ Hueco 4: Recorta la camiseta y agrupa sus colores

Recorta la caja del jugador del frame, quédate con la **mitad superior** (la camiseta) y agrupa sus píxeles con K-Means.

<details><summary>💡 Pista</summary>

Una imagen en NumPy se indexa `imagen[filas, columnas]`, es decir `[y, x]`: `frames[0][y1:y2, x1:x2]`. La altura del recorte es `crop.shape[0]`; la mitad es `crop.shape[0]//2`. ¿Cuántos colores esperas en la mitad superior? Camiseta y pasto.

</details>

In [ ]:
from sklearn.cluster import KMeans

player_id, player = next(iter(tracks["players"][0].items()))
x1, y1, x2, y2 = map(int, player["bbox"])
crop = frames[0][_____, _____]     # ✏️ HUECO 4a: recorta la caja: primero filas (y), luego columnas (x)
top_half = crop[: _____]         # ✏️ HUECO 4b: solo la mitad superior (la camiseta)

kmeans = KMeans(n_clusters=_____, n_init=1).fit(top_half.reshape(-1, 3))   # ✏️ HUECO 4c: ¿cuántos grupos de color?
clustered = kmeans.labels_.reshape(top_half.shape[:2])

big = lambda img: cv2.resize(img, None, fx=6, fy=6, interpolation=cv2.INTER_NEAREST)
show(big(crop), big(top_half), big((clustered*255).astype(np.uint8)),
     titles=[f"Jugador {player_id}", "Mitad superior", "2 clusters de color"], width=10)

In [ ]:
# Verificación del hueco 4
revisar(4, lambda: crop.shape[:2] == (y2 - y1, x2 - x1) and top_half.shape[0] == crop.shape[0] // 2 and kmeans.n_clusters == 2, 'Revisa el orden [y, x] del recorte, la mitad con // y el número de clusters.')

<details><summary>👀 Ver solución del hueco 4</summary>

```python
crop = frames[0][y1:y2, x1:x2]
top_half = crop[: crop.shape[0]//2]

kmeans = KMeans(n_clusters=2, n_init=1).fit(top_half.reshape(-1, 3))
```

</details>

**Paso B:** juntamos el color de camiseta de todos los jugadores (en 10 frames) y volvemos a usar K-Means con
2 grupos: ¡los equipos!

In [ ]:
from team_assigner import TeamAssigner

team_assigner = TeamAssigner()
team_assigner.assign_team_color(frames, tracks["players"])

sample = [team_assigner.get_player_color(frames[0], p["bbox"]) for p in tracks["players"][0].values()]
sample = np.array(sample)
plt.figure(figsize=(6, 5))
plt.scatter(sample[:, 2], sample[:, 1], c=sample[:, ::-1]/255, s=150, edgecolors="k")
for team, color in team_assigner.team_colors.items():
    plt.scatter(color[2], color[1], marker="X", s=400, c=[color[::-1]/255], edgecolors="k", label=f"Centro equipo {team}")
plt.xlabel("Rojo"); plt.ylabel("Verde"); plt.legend(); plt.title("Color de camiseta de cada jugador (frame 0)")
plt.show()

**Paso C: corregir a YOLO con el color.** El modelo a veces marca a un jugador como `referee` durante
muchos frames seguidos (el voto por mayoría de la sección 3 no lo arregla porque el error es consistente).
Pero ya conocemos los colores de los equipos: si la camiseta de un "árbitro" es casi igual a la de un equipo,
es un jugador. Los árbitros (negro) y jueces de línea quedan lejos de ambos colores.

In [ ]:
moved = team_assigner.fix_referees(frames, tracks, max_ratio=config.REFEREE_TEAM_COLOR_RATIO)
print(f"Tracks de 'árbitro' que en realidad son jugadores: {sorted(moved)}")

**Paso D: equipo de cada jugador en cada frame.** Cada 5 frames volvemos a medir el color de la camiseta y el
equipo es la mayoría de las últimas 9 mediciones. ¿Por qué no fijarlo una vez por ID? Porque el tracker a veces
le pasa el ID de un jugador a otro (*ID switch*), y ese ID puede empezar en un equipo y terminar en el otro.

### ✏️ Hueco 5: Guarda el equipo de cada jugador

`get_player_team` ya te dice el equipo (1 o 2) del jugador en ese frame. Guárdalo en el track junto con el color del equipo, para poder dibujarlo.

<details><summary>💡 Pista</summary>

`team_assigner.team_colors` es un diccionario `{equipo: color}`: el color del equipo `team` es `team_assigner.team_colors[team]`.

</details>

In [ ]:
for frame_num, player_track in enumerate(tracks["players"]):
    for player_id, track in player_track.items():
        team = team_assigner.get_player_team(frames[frame_num], track["bbox"], player_id)
        track["team"] = _____                                   # ✏️ HUECO 5a: el equipo del jugador
        track["team_color"] = team_assigner.team_colors[_____]  # ✏️ HUECO 5b: el color de ese equipo

def draw_teams(frame_num):
    frame = frames[frame_num].copy()
    for track_id, player in tracks["players"][frame_num].items():
        tracker.draw_ellipse(frame, player["bbox"], player["team_color"], track_id)
    return frame

frame_b = min(300, len(frames) - 1)
show(draw_teams(150), draw_teams(frame_b), titles=["Frame 150", f"Frame {frame_b}"])


In [ ]:
# Verificación del hueco 5
revisar(5, lambda: all(p["team"] in (1, 2) and (p["team_color"] == team_assigner.team_colors[p["team"]]).all() for f in tracks["players"] for p in f.values()), 'Cada jugador debe tener team (1 o 2) y el team_color de ese equipo.')

<details><summary>👀 Ver solución del hueco 5</summary>

```python
        track["team"] = team
        track["team_color"] = team_assigner.team_colors[team]
```

</details>

🧪 **Retos:**
- ¿Qué pasa con los **porteros**? Su camiseta es de otro color. ¿A qué equipo los asigna? (Lo resolvemos en la sección 6.)
- ¿Funcionaría igual un partido donde ambos equipos visten colores parecidos?

## 5. El balón y la posesión · ~15 min

El balón es pequeño y rápido: YOLO lo pierde en muchos frames. Como su trayectoria es suave,
podemos **interpolar** las posiciones faltantes... pero solo en huecos cortos (< 0.7 s). En un hueco largo el
balón pudo ir a cualquier lado, y una línea recta dibujaría el marcador lejos del balón real.

In [ ]:
import copy
raw_ball = copy.deepcopy(tracks["ball"])   # detecciones originales: las limpiamos en la sección 6

ball_x_raw = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"], max_gap=config.BALL_MAX_GAP_FRAMES)
tracker.add_position_to_tracks(tracks)   # recalcular posiciones con el balon interpolado
ball_x = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]

print(f"Frames sin balón detectado: {np.isnan(ball_x_raw).sum()} de {len(ball_x_raw)}")
plt.figure(figsize=(12, 3))
plt.plot(ball_x, label="interpolado", color="orange")
plt.plot(ball_x_raw, ".", markersize=3, label="detectado por YOLO", color="k")
plt.xlabel("frame"); plt.ylabel("x del balón (px)"); plt.legend(); plt.show()

**Posesión:** en cada frame, el balón es del jugador cuyo pie está más cerca (máximo `config.MAX_PLAYER_BALL_DISTANCE` px).
Si nadie está cerca (pase en el aire), la posesión sigue siendo del último equipo que la tuvo.

### ✏️ Hueco 6: ¿Quién tiene el balón?

Completa la lógica de posesión. Para cada frame agregamos a `control` el equipo que tiene el balón:
- Si hay un jugador cerca del balón (`assigned != -1`), es **su** equipo.
- Si nadie está cerca (pase en el aire), se queda el **último** equipo que lo tuvo; si todavía no lo ha tenido nadie, ponemos `0`.

<details><summary>💡 Pista</summary>

El equipo del jugador asignado está en `player_track[assigned]["team"]`. El último elemento de una lista es `control[-1]`; para el caso vacío usa `control[-1] if control else 0`.

</details>

In [ ]:
from player_ball_assigner import PlayerBallAssigner

player_assigner = PlayerBallAssigner(max_player_ball_distance=config.MAX_PLAYER_BALL_DISTANCE)

def compute_possession():
    control = []
    for frame_num, player_track in enumerate(tracks["players"]):
        for player in player_track.values():
            player["has_ball"] = False
        ball = tracks["ball"][frame_num].get(1)
        assigned = player_assigner.assign_ball_to_player(player_track, ball["bbox"]) if ball else -1
        if assigned != -1:
            player_track[assigned]["has_ball"] = True
            control.append(_____)   # ✏️ HUECO 6a: el equipo del jugador que tiene el balón
        else:
            control.append(_____)   # ✏️ HUECO 6b: el último equipo que lo tuvo (o 0 si la lista está vacía)
    control = np.array(control)

    for team in (1, 2):
        print(f"Equipo {team}: {np.mean(control[control>0]==team)*100:.1f}% de posesión")
    colors = {t: c[::-1]/255 for t, c in team_assigner.team_colors.items()}
    plt.figure(figsize=(12, 1.5))
    plt.scatter(range(len(control)), np.zeros(len(control)),
                c=[colors.get(t, (0.8, 0.8, 0.8)) for t in control], marker="|", s=800)
    plt.yticks([]); plt.xlabel("frame"); plt.title("¿Quién tiene el balón?"); plt.show()
    return control

team_ball_control = compute_possession()

In [ ]:
def _check_posesion():
    c = list(team_ball_control)
    if len(c) != len(frames) or not set(c) <= {0, 1, 2}:
        return False
    for i, f in enumerate(tracks["players"]):
        dueño = [p["team"] for p in f.values() if p.get("has_ball")]
        esperado = dueño[0] if dueño else (c[i - 1] if i else 0)
        if c[i] != esperado:
            return False
    return True

# Verificación del hueco 6
revisar(6, lambda: _check_posesion(), 'Cuando nadie tiene el balón, el valor debe ser el del frame anterior.')

<details><summary>👀 Ver solución del hueco 6</summary>

```python
            control.append(player_track[assigned]["team"])
        else:
            control.append(control[-1] if control else 0)
```

</details>

🧪 **Reto:** cambia `max_player_ball_distance` a 30 y a 150. ¿Cómo cambia la posesión? ¿Qué valor te parece más realista?

## 6. Bonus: de píxeles a metros · ~15 min

Para medir **velocidad** necesitamos metros, no píxeles. Dos problemas:

1. **La cámara se mueve**: hace paneo y zoom para seguir la jugada. Con *optical flow* seguimos
   cientos de puntos entre frames consecutivos y calculamos la **homografía** que lleva un frame al anterior
   (RANSAC ignora los puntos que se mueven solos, como los jugadores). Encadenándolas, cualquier frame se
   puede llevar al **frame de referencia** (`config.CALIBRATION_FRAME`, el frame 380 en este clip).
2. **La perspectiva**: 1 px cerca de la cámara no mide lo mismo que 1 px al fondo. En el frame de referencia
   calculamos otra homografía, de píxeles a metros, con puntos conocidos de la cancha: línea de medio campo,
   bandas y círculo central (radio 9.15 m).
3. **Quién está en la cancha**: con la posición en metros quitamos a las personas **detrás de la valla**
   publicitaria (stewards, fotógrafos). La máscara de pasto de la sección 3 no basta, porque YOLO a veces dibuja
   su caja hasta el pasto. En metros es fácil: todo lo que esté a más de 3 m de las líneas, fuera.

In [ ]:
from camera_movement_estimator import CameraMovementEstimator

camera_estimator = CameraMovementEstimator(frames[0], overlay_boxes=config.OVERLAY_BOXES,
                                           text_box=config.CAMERA_MOVEMENT_BOX,
                                           reference_frame=config.CALIBRATION_FRAME)
camera_movement = camera_estimator.get_camera_movement(frames, read_from_stub=USE_STUBS,
                                                       stub_path=config.CAMERA_STUB_PATH)
camera_estimator.add_adjust_positions_to_tracks(tracks, camera_movement)

check_frame = min(300, len(frames) - 1)
print(f"Homografía del frame {check_frame} al frame de referencia:")
print(camera_estimator.to_reference(camera_movement, check_frame).round(3))

summary = camera_estimator.get_camera_summary(camera_movement, frames[0].shape)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
ax1.plot(summary[:, 0], label="X"); ax1.plot(summary[:, 1], label="Y")
ax1.set_xlabel("frame"); ax1.set_ylabel("px"); ax1.set_title("Paneo respecto al frame de referencia"); ax1.legend()
ax2.plot(summary[:, 2], color="purple")
ax2.set_xlabel("frame"); ax2.set_title("Zoom respecto al frame de referencia")
plt.show()


In [ ]:
# Frame de referencia: donde se calibró
!{sys.executable} tools/calibrate_pitch.py --check
# Otro frame del mismo clip: la cámara está en otra zona y con otro zoom; la cancha se dibuja
# siguiendo las homografías encadenadas (elegimos uno lejos de CALIBRATION_FRAME pero dentro del clip)
other_frame = max(0, config.CALIBRATION_FRAME - 250)
!{sys.executable} tools/calibrate_pitch.py --check --frame {other_frame} --output output_videos/calibration_check_other.jpg
show(cv2.imread("output_videos/calibration_check.jpg"), cv2.imread("output_videos/calibration_check_other.jpg"),
     titles=[f"Frame {config.CALIBRATION_FRAME} (calibración)", f"Frame {other_frame} (siguiendo a la cámara)"])


In [ ]:
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator

view_transformer = ViewTransformer(config.PITCH_POINTS_PX, config.PITCH_POINTS_M,
                                   config.PITCH_LENGTH_M, config.PITCH_WIDTH_M,
                                   margin=config.PITCH_MARGIN_M)
view_transformer.add_transformed_position_to_tracks(tracks)

# Ya sabemos dónde está cada persona en metros: fuera quien esté a más de 3 m de la cancha
# (stewards, fotógrafos y recogebalones detrás de la valla publicitaria)
print("Detecciones fuera de la cancha eliminadas:", tracker.remove_off_pitch(tracks))

### El balón que no era balón

Mira la gráfica del balón de la sección 5: hay saltos imposibles. YOLO a veces confunde con el balón el
**punto penal** (una mancha blanca en el pasto) o algo en el borde de la imagen, y como nos quedábamos con la
detección de mayor confianza, el marcador saltaba de un lado a otro de la cancha.

Ahora que tenemos metros, usamos la **física**: de todas las candidatas de cada frame nos quedamos con la más
cercana a la última posición del balón, y solo si pudo llegar ahí (un disparo fuerte anda por 30-40 m/s).

In [ ]:
def ball_to_meters(frame_num, point):
    H = camera_estimator.to_reference(camera_movement, frame_num)
    return view_transformer.pixel_to_meters(point, H)

tracks["ball"] = copy.deepcopy(raw_ball)
print("Detecciones descartadas:", tracker.select_ball(tracks, ball_to_meters, fps))
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"], max_gap=config.BALL_MAX_GAP_FRAMES)
tracker.add_position_to_tracks(tracks)

ball_x_clean = [f[1]["bbox"][0] if 1 in f else np.nan for f in tracks["ball"]]
plt.figure(figsize=(12, 3))
plt.plot(ball_x_raw, ".", markersize=3, color="lightgray", label="YOLO (mayor confianza)")
plt.plot(ball_x_clean, color="orange", label="siguiendo la trayectoria")
plt.xlabel("frame"); plt.ylabel("x del balón (px)"); plt.legend(); plt.show()

### ¿De qué equipo es el portero?

El portero no viste los colores de su equipo, así que K-Means lo manda al equipo con el color más parecido
(a veces el rival). Tampoco basta con mirar a los jugadores cercanos: en un contragolpe o un córner lo rodean rivales.

La idea: responder dos preguntas que **no dependen de un solo frame**:
1. **¿Qué portería defiende cada equipo?** En cada frame, el equipo que defiende la portería derecha suele estar
   más a la derecha que el rival. Los contragolpes invierten unos pocos frames, pero votamos con *todo* el video.
2. **¿De qué lado está cada portero?** YOLO distingue la clase `goalkeeper` (no en todos los frames); guardamos esa marca en el track, descartamos los que visten el color de un equipo y
   tomamos la **mediana** de su posición en metros. Un portero casi nunca sale de su tercio de cancha.

In [ ]:
goalkeeper_teams = team_assigner.assign_goalkeepers(frames, tracks)
confidence = max(team_assigner.side_vote, 1 - team_assigner.side_vote)
print(f"El equipo {team_assigner.team_right} defiende la portería derecha en el {confidence*100:.0f}% de los frames")
print("Porteros (track: equipo):", goalkeeper_teams)

# Con el balón limpio, los porteros corregidos y sin la gente de fuera de la cancha, recalculamos la posesión
team_ball_control = compute_possession()

In [ ]:
speed_estimator = SpeedAndDistance_Estimator(frame_rate=fps, max_speed_kmh=config.MAX_PLAYER_SPEED_KMH)
speed_estimator.add_speed_and_distance_to_tracks(tracks)

In [ ]:
# Vista "radar": posiciones reales en la cancha en un frame
def draw_radar(frame_num, ax):
    ax.set_facecolor("#3a7d44")
    ax.plot([0, 0], [0, 68], "w"); ax.add_patch(plt.Circle((0, 34), 9.15, fill=False, color="w"))
    for team_key in ("players", "referees"):
        for tid, t in tracks[team_key][frame_num].items():
            if t.get("position_transformed") is None:
                continue
            x, y = t["position_transformed"]
            color = t["team_color"][::-1]/255 if "team_color" in t else "white"
            ax.scatter(x, y, color=color, s=120, edgecolors="k")
    ax.set_xlim(-52.5, 52.5); ax.set_ylim(68, 0); ax.set_aspect("equal")
    ax.set_title(f"Frame {frame_num}"); ax.set_xticks([]); ax.set_yticks([])

# 3 frames repartidos a lo largo del clip actual (evita indices fuera de rango si el clip es corto)
sample_frames = sorted(set(min(n, len(frames) - 1) for n in (50, len(frames)//2, len(frames) - 10)))
fig, axes = plt.subplots(1, len(sample_frames), figsize=(16, 4))
for ax, n in zip(axes, sample_frames):
    draw_radar(n, ax)
plt.show()


### ✏️ Hueco 7: Ciencia de datos: estadísticas por jugador

Con pandas, agrupa las filas **por jugador** (`id`) y calcula su **velocidad máxima**. La distancia ya está calculada: como es *acumulada*, su máximo es el total recorrido.

<details><summary>💡 Pista</summary>

`groupby` recibe el nombre de la columna por la que agrupas (`"id"`). Para el máximo, `agg` acepta el texto `"max"`.

</details>

In [ ]:
import pandas as pd

rows = []
for frame in tracks["players"]:
    for tid, t in frame.items():
        if "speed" in t:
            rows.append({"id": tid, "equipo": t["team"], "km/h": t["speed"], "metros": t["distance"]})

stats = pd.DataFrame(rows).groupby(_____).agg(equipo=("equipo", "first"),   # ✏️ HUECO 7a: agrupa por jugador
                                             vel_max=("km/h", _____),     # ✏️ HUECO 7b: la velocidad máxima
                                             distancia=("metros", "max"))
stats.sort_values("vel_max", ascending=False).head(10).round(1)

In [ ]:
# Verificación del hueco 7
revisar(7, lambda: stats.index.name == "id" and (stats["vel_max"] == pd.DataFrame(rows).groupby("id")["km/h"].max()).all(), 'Agrupa por "id" y usa "max" para la velocidad.')

<details><summary>👀 Ver solución del hueco 7</summary>

```python
stats = pd.DataFrame(rows).groupby("id").agg(equipo=("equipo", "first"),
                                             vel_max=("km/h", "max"),
```

</details>

⚠️ **Limitaciones (para discutir):** encadenar homografías acumula error (*drift*): tras miles de frames la cancha
proyectada se desvía unos metros. La solución profesional es detectar los puntos clave de la cancha en *cada*
frame con otro modelo (ver [roboflow/sports](https://github.com/roboflow/sports)).

## 🎬 Video final

Juntamos todo en un video: elipses por equipo, balón, posesión, movimiento de cámara y, abajo al centro, un **mini-mapa** con la posición real (en metros) de cada jugador. Las banderas `DRAW_*` encienden o apagan cada capa.

In [ ]:
# Dibujamos sobre los frames originales (ya no los necesitamos limpios) y guardamos el video
from utils import save_video
from mini_map import MiniMapDrawer

DRAW_IDS = config.DRAW_TRACK_IDS          # True/False: número de cada jugador
DRAW_SPEED = config.DRAW_SPEED_DISTANCE   # True/False: km/h y metros recorridos
DRAW_MINIMAP = config.MINI_MAP           # True/False: mini-mapa (radar) de la cancha abajo al centro

output = tracker.draw_annotations(frames, tracks, team_ball_control, team_assigner.team_colors, draw_ids=DRAW_IDS)
output = camera_estimator.draw_camera_movement(output, camera_movement)
if DRAW_SPEED:
    speed_estimator.draw_speed_and_distance(output, tracks)
if DRAW_MINIMAP:
    # Usa position_transformed (metros) de cada jugador; en amarillo el que tiene el balón
    mini_map = MiniMapDrawer(pitch_length=config.PITCH_LENGTH_M, pitch_width=config.PITCH_WIDTH_M,
                             box=config.MINI_MAP_BOX, bg_color=config.MINI_MAP_BG_COLOR,
                             bg_alpha=config.MINI_MAP_BG_ALPHA, line_color=config.MINI_MAP_LINE_COLOR,
                             player_radius=config.MINI_MAP_PLAYER_RADIUS,
                             ball_carrier_color=config.MINI_MAP_BALL_CARRIER_COLOR)
    output = mini_map.draw_mini_map(output, tracks)
save_video(output, config.OUTPUT_PATH, fps=fps)
show_video(config.OUTPUT_PATH)

## 7. Retos para seguir · ~10 min

1. **Mapa de calor** de un jugador usando `position_transformed` y `plt.hist2d`.
2. **Radar animado**: convierte `draw_radar` en un video lado a lado con la transmisión.
3. **Otro clip**: usa `tools/find_wide_shots.py` para encontrar otra toma abierta, recórtala y recalibra con
   `tools/calibrate_pitch.py`.
4. **Entrena tu propio modelo** con YOLO11 y el dataset de Roboflow (sección 2) y compáralo con `best.pt`.
5. **Porteros**: asígnalos al equipo cuya mitad de cancha defienden, en vez de usar el color.

### Créditos
- Código base: [abdullahtarek/football_analysis](https://github.com/abdullahtarek/football_analysis) (el autor tiene un tutorial completo en YouTube)
- Dataset: [Roboflow football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc)
- Modelos y librerías: [Ultralytics YOLO](https://docs.ultralytics.com), [supervision](https://supervision.roboflow.com)
- Video: transmisión de TVP, final de la Copa Mundial de la FIFA Qatar 2022 (uso educativo)